## Reproducibility & AI-assistance disclosure

Part of **`dislocation-speech-translation-fr-en`**, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by N. Ballier).

**AI-assistance disclosure.** The executable code is the author's. The explanatory **comments and markdown** were drafted with the help of a large language model (LLM) and reviewed by the author; the LLM changed no executable logic.

**Pipeline order:** `01_whisper` → `02_xlsr` → `03_align` → `04_score`.


# Dislocation segment evaluation — BLEU, chrF, TER, COMET

Computes per-segment automatic evaluation scores for the three model outputs
(Whisper sliced, Whisper continuous, XLS-R) against two reference translations
(structure-preserving calque and idiomatic SVO), on the 79 annotated dislocation segments.

**Input:** `dislocations_aligned_FINAL.xlsx` — one row per dislocation, all model outputs
and reference translations already aligned.

**Output:** `disloc_eval_scores.xlsx` — same rows with 24 score columns appended
(4 metrics × 3 models × 2 references).

**Metrics:**
- BLEU (`effective_order=True`, `lowercase=True`), chrF (defaults), TER — via `sacrebleu`
- COMET — `Unbabel/wmt22-comet-da` via `unbabel-comet`
  - Source field: `Citation contexte (FR)` (the annotated French dislocation)
  - MT field: model hypothesis
  - Ref field: reference translation (scored separately against CALQUE and IDIOMATIC)

**Runtime:** GPU recommended for COMET (free Colab T4 is sufficient).
Surface metrics (BLEU/chrF/TER) run fast on CPU.

## Cell 1 — Install dependencies

Run once, then **restart the runtime** (Runtime › Restart session) before proceeding.
After restarting, skip this cell and run from Cell 2 onward.

In [ ]:
# Install scoring deps, then RESTART the runtime before continuing (numpy/protobuf pins).

!pip install unbabel-comet sacrebleu -q
!pip install "numpy>=2.0" "protobuf>=5.0" -q
print("Installation complete. Restart the runtime before continuing.")

## Cell 2 — Configuration

Mount your Drive (or use `files.upload()`) then set the three paths below.
Column names should not need editing unless you renamed columns in the input file.

In [ ]:
# Config. INPUT_FILE = dislocation_translations.csv. HYPOTHESES keys (whisper_per_seg / whisper_cont /
# xlsr) define the score-column names; COMET source = the FR citation; COMET = Unbabel/wmt22-comet-da.

# --- Google Drive (recommended) ---
from google.colab import drive
drive.mount('/content/drive')

# --- Direct upload (alternative) ---
# from google.colab import files
# files.upload()   # upload dislocations_aligned_FINAL.xlsx

# ── Paths ──────────────────────────────────────────────────────────────
INPUT_FILE  = 'data/dislocation_translations.csv'
OUTPUT_FILE = 'data/dislocation_scores.csv'

# ── Hypotheses: {score_label: column_name_in_xlsx} ─────────────────────
HYPOTHESES = {
    'whisper_per_seg': 'whisper_en_per_seg',  # Whisper per-segment
    'whisper_cont':    'whisper_en_cont',     # Whisper continuous (re-indexed via SRT)
    'xlsr':            'xlsr_en',             # XLS-R direct ST
}

# ── References: {score_label: column_name_in_xlsx} ─────────────────────
REFERENCES = {
    'calque':    'Référence trad. CALQUE (EN)',    # structure-preserving
    'idiomatic': 'Référence trad. IDIOMATIC (EN)', # canonical SVO
}

# ── COMET source column (French dislocation citation) ───────────────────
COL_SRC = 'Citation contexte (FR)'

# ── COMET model ─────────────────────────────────────────────────────────
COMET_MODEL = 'Unbabel/wmt22-comet-da'

print('Configuration loaded.')
print(f'  Hypotheses : {list(HYPOTHESES.keys())}')
print(f'  References : {list(REFERENCES.keys())}')

## Cell 3 — Load data

In [ ]:
# Load the aligned table and assert all expected columns are present.

import pandas as pd

df = pd.read_csv(INPUT_FILE)
print(f'Rows loaded : {len(df)}')
print(f'Columns     : {list(df.columns)}')

# Verify all expected columns are present
expected = [COL_SRC] + list(HYPOTHESES.values()) + list(REFERENCES.values())
missing  = [c for c in expected if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')
print('All expected columns found.')

## Cell 4 — Normalise and validate

Cast all text columns to `str`, strip whitespace, replace `'nan'` strings
with empty strings. Report empty cells per column.

In [ ]:
# Normalise text columns and report empty cells (empties become NaN, not zero, downstream).

def _norm(series):
    return series.astype(str).replace({'nan': ''}).str.strip()

for col in [COL_SRC] + list(HYPOTHESES.values()) + list(REFERENCES.values()):
    df[col] = _norm(df[col])

print('Empty cells per column:')
for col in [COL_SRC] + list(HYPOTHESES.values()) + list(REFERENCES.values()):
    n = (df[col] == '').sum()
    flag = '  ← WARNING' if n > 0 else ''
    print(f'  {col}: {n} empty{flag}')

## Cell 5 — Surface metrics: BLEU, chrF, TER

Scored sentence-by-sentence for every (hypothesis, reference) pair.
Column naming: `{metric}_{model}_{ref}` — e.g. `bleu_xlsr_en_calque`.

- `effective_order=True`: prevents zero BLEU on short segments with no 4-gram matches
- `lowercase=True`: case-insensitive matching (both refs are already lowercase/punctuation-free)
- TER is an **error** metric — lower is better
- Empty hypothesis or reference → `NaN`

In [ ]:
# Surface metrics via sacrebleu: BLEU(effective_order=True, lowercase=True), CHRF (defaults), TER;
# per hypothesis x reference pair; empties -> NaN.

from sacrebleu.metrics import BLEU, CHRF, TER

bleu_metric = BLEU(effective_order=True, lowercase=True)
chrf_metric = CHRF()  # char_order=6, word_order=0, beta=2 (sacrebleu defaults)
ter_metric  = TER()

def score_surface(hyp, ref):
    """Return (BLEU, chrF, TER) for one pair. NaN if either is empty."""
    if not hyp or not ref:
        return float('nan'), float('nan'), float('nan')
    b = bleu_metric.sentence_score(hyp, [ref]).score
    c = chrf_metric.sentence_score(hyp, [ref]).score
    t = ter_metric.sentence_score(hyp,  [ref]).score
    return round(b, 4), round(c, 4), round(t, 4)

surface_score_cols = []

for hyp_key, hyp_col in HYPOTHESES.items():
    for ref_key, ref_col in REFERENCES.items():
        bleu_col = f'bleu_{hyp_key}_{ref_key}'
        chrf_col = f'chrf_{hyp_key}_{ref_key}'
        ter_col  = f'ter_{hyp_key}_{ref_key}'
        surface_score_cols += [bleu_col, chrf_col, ter_col]

        bleu_scores, chrf_scores, ter_scores = [], [], []
        for hyp, ref in zip(df[hyp_col], df[ref_col]):
            b, c, t = score_surface(hyp, ref)
            bleu_scores.append(b)
            chrf_scores.append(c)
            ter_scores.append(t)

        df[bleu_col] = bleu_scores
        df[chrf_col] = chrf_scores
        df[ter_col]  = ter_scores
        print(f'  Scored: {hyp_key} vs {ref_key}')

print(f'\nSurface scoring complete — {len(surface_score_cols)} score columns added.')

## Cell 6 — Load COMET model

Downloads `Unbabel/wmt22-comet-da` (~1.5 GB) the first time; cached on subsequent runs.
Requires a GPU runtime for reasonable speed (Runtime › Change runtime type › T4).

In [ ]:
# Download and load the COMET checkpoint (Unbabel/wmt22-comet-da).

from comet import download_model, load_from_checkpoint

model_path  = download_model(COMET_MODEL)
comet_model = load_from_checkpoint(model_path)
print('COMET model loaded.')

## Cell 7 — COMET scores

Scored for every (hypothesis, reference) pair.
Source field = `Citation contexte (FR)` (annotated French dislocation).
Rows where source, hypothesis, or reference is empty receive `NaN`.

Column naming: `comet_{model}_{ref}` — e.g. `comet_xlsr_en_idiomatic`.

In [ ]:
# COMET per hypothesis x reference: src=FR citation, mt=output, ref=reference; empty triples skipped.

comet_score_cols = []

for hyp_key, hyp_col in HYPOTHESES.items():
    for ref_key, ref_col in REFERENCES.items():
        comet_col = f'comet_{hyp_key}_{ref_key}'
        comet_score_cols.append(comet_col)
        df[comet_col] = float('nan')

        data, indices = [], []
        for i, row in df.iterrows():
            src = row[COL_SRC]
            mt  = row[hyp_col]
            ref = row[ref_col]
            if src and mt and ref:
                data.append({'src': src, 'mt': mt, 'ref': ref})
                indices.append(i)

        if not data:
            print(f'  [{hyp_key} vs {ref_key}] No valid rows — skipping.')
            continue

        print(f'  [{hyp_key} vs {ref_key}] Scoring {len(data)} segments...')
        output = comet_model.predict(data, batch_size=8, gpus=1)
        # gpus=0 if running on CPU

        for idx, score in zip(indices, output.scores):
            df.at[idx, comet_col] = round(score, 4)

print(f'\nCOMET scoring complete — {len(comet_score_cols)} score columns added.')

## Cell 8 — Summary statistics

Mean scores per model, broken down by reference (CALQUE vs IDIOMATIC).
TER is an error metric — lower is better for TER, higher is better for the others.

In [ ]:
# Console summary: mean BLEU/chrF/TER/COMET per hypothesis x reference, plus NaN counts.

import warnings
warnings.filterwarnings('ignore')

print('=' * 65)
print(f'  Mean scores — 79 dislocation segments')
print('=' * 65)
print(f'{"":<20} {"BLEU":>8} {"chrF":>8} {"TER":>8} {"COMET":>8}')
print('-' * 65)

for hyp_key in HYPOTHESES:
    for ref_key in REFERENCES:
        b = df[f'bleu_{hyp_key}_{ref_key}'].mean()
        c = df[f'chrf_{hyp_key}_{ref_key}'].mean()
        t = df[f'ter_{hyp_key}_{ref_key}'].mean()
        k = df[f'comet_{hyp_key}_{ref_key}'].mean()
        label = f'{hyp_key} / {ref_key}'
        print(f'{label:<20} {b:>8.2f} {c:>8.2f} {t:>8.2f} {k:>8.4f}')
    print('-' * 65)

all_score_cols = surface_score_cols + comet_score_cols
print(f'\nTotal score columns: {len(all_score_cols)}')
print(f'NaN counts per column:')
for col in all_score_cols:
    n = df[col].isna().sum()
    if n > 0:
        print(f'  {col}: {n} NaN')

## Cell 9 — Export to Excel

In [ ]:
# Export the 79 x 24 score matrix to dislocation_scores.csv.


# Columns to include in output: identifiers + texts + all scores
id_cols   = ['ID', 'Speaker', 'Type', 'n_segs', 'segments', 'audio_start_s']
text_cols = [COL_SRC,
             'Référence trad. CALQUE (EN)',
             'Référence trad. IDIOMATIC (EN)',
             'whisper_en_per_seg',
             'whisper_en_cont',
             'xlsr_en']
all_score_cols = surface_score_cols + comet_score_cols

out_cols = [c for c in id_cols + text_cols if c in df.columns] + all_score_cols
df_out = df[out_cols].copy()

df_out.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f'Exported: {OUTPUT_FILE}')
print(f'Shape: {df_out.shape}')
print(f'Columns: {list(df_out.columns)}')